### Import Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

### Load Dataset

In [2]:
dataset=pd.read_csv("Amazon.csv")

In [3]:
dataset.head()

,Product Name,Brand Name,Price,Rating,Reviews,Review Votes
0,"""CLEAR CLEAN ESN"" Sprint EPIC 4G Galaxy SPH-D7...",Samsung,199.99,5,I feel so LUCKY to have found this used (phone...,1.0
1,"""CLEAR CLEAN ESN"" Sprint EPIC 4G Galaxy SPH-D7...",Samsung,199.99,4,"nice phone, nice up grade from my pantach revu...",0.0
2,"""CLEAR CLEAN ESN"" Sprint EPIC 4G Galaxy SPH-D7...",Samsung,199.99,5,Very pleased,0.0
3,"""CLEAR CLEAN ESN"" Sprint EPIC 4G Galaxy SPH-D7...",Samsung,199.99,4,It works good but it goes slow sometimes but i...,0.0
4,"""CLEAR CLEAN ESN"" Sprint EPIC 4G Galaxy SPH-D7...",Samsung,199.99,4,Great phone to replace my lost phone. The only...,0.0


### Clean Dataset

In [4]:
dataset = dataset[['Product Name', 'Brand Name', 'Reviews']]

In [5]:
dataset.head()

,Product Name,Brand Name,Reviews
0,"""CLEAR CLEAN ESN"" Sprint EPIC 4G Galaxy SPH-D7...",Samsung,I feel so LUCKY to have found this used (phone...
1,"""CLEAR CLEAN ESN"" Sprint EPIC 4G Galaxy SPH-D7...",Samsung,"nice phone, nice up grade from my pantach revu..."
2,"""CLEAR CLEAN ESN"" Sprint EPIC 4G Galaxy SPH-D7...",Samsung,Very pleased
3,"""CLEAR CLEAN ESN"" Sprint EPIC 4G Galaxy SPH-D7...",Samsung,It works good but it goes slow sometimes but i...
4,"""CLEAR CLEAN ESN"" Sprint EPIC 4G Galaxy SPH-D7...",Samsung,Great phone to replace my lost phone. The only...


In [6]:
# remove null values
dataset.dropna(inplace=True)
dataset.reset_index(drop=True, inplace=True)

In [7]:
# rename columns
dataset.columns = ['Product', 'Brand', 'Review']
dataset.head()

,Product,Brand,Review
0,"""CLEAR CLEAN ESN"" Sprint EPIC 4G Galaxy SPH-D7...",Samsung,I feel so LUCKY to have found this used (phone...
1,"""CLEAR CLEAN ESN"" Sprint EPIC 4G Galaxy SPH-D7...",Samsung,"nice phone, nice up grade from my pantach revu..."
2,"""CLEAR CLEAN ESN"" Sprint EPIC 4G Galaxy SPH-D7...",Samsung,Very pleased
3,"""CLEAR CLEAN ESN"" Sprint EPIC 4G Galaxy SPH-D7...",Samsung,It works good but it goes slow sometimes but i...
4,"""CLEAR CLEAN ESN"" Sprint EPIC 4G Galaxy SPH-D7...",Samsung,Great phone to replace my lost phone. The only...


In [8]:
# 3. Remove Duplicates
dataset = dataset.drop_duplicates(subset='Product')
dataset.reset_index(drop=True, inplace=True)

### Combine Text

In [9]:
dataset['combined'] = dataset['Product'] + " " + dataset['Brand'] + " " + dataset['Review']

### Text Preprocessing

In [10]:
import re
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\saniy\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [11]:
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

In [12]:
ps = PorterStemmer()
all_stopwords = stopwords.words('english')
all_stopwords.remove('not')

In [13]:
corpus = []

In [14]:
for i in range(len(dataset)):
    text = re.sub('[^a-zA-Z]', ' ', dataset['combined'].iloc[i])
    text = text.lower().split()
    
    text = [ps.stem(word) for word in text if word not in all_stopwords]
    text = ' '.join(text)
    
    corpus.append(text)

### Vectorization (TF- IDF)

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [16]:
cv = TfidfVectorizer(max_features=3000)
vectors = cv.fit_transform(corpus) 

### Similarity Matrix

In [17]:
from sklearn.metrics.pairwise import cosine_similarity

In [18]:
def recommend(product_name):

    if product_name not in dataset['Product'].values:
        print("❌ Product not found")
        return

    index = dataset[dataset['Product'] == product_name].index[0]
    brand = dataset.iloc[index]['Brand']

    distances = cosine_similarity(vectors[index], vectors).flatten()

    product_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1]
    )

    print("\n💡 Recommended Products:\n")

    shown = 0
    used = set()

    for i in product_list:
        prod = dataset.iloc[i[0]]

        if prod['Brand'] == brand and prod['Product'] != product_name:
            print(prod['Product'])
            used.add(prod['Product'])
            shown += 1

        if shown == 5:
            return

    for i in product_list:
        prod = dataset.iloc[i[0]]

        if prod['Product'] != product_name and prod['Product'] not in used:
            print(prod['Product'])
            shown += 1

        if shown == 5:
            break

In [19]:
def product_system(query):

    query = query.lower()

    results = dataset[
        dataset['Product'].str.lower().str.contains(query) |
        dataset['Brand'].str.lower().str.contains(query)
    ]

    if results.empty:
        print("❌ No products found")
        return

    # show options
    print("\n🔍 Matching Products:\n")

    results = results.head(5).reset_index(drop=True)

    for i, row in results.iterrows():
        print(f"{i} | {row['Brand']} | {row['Product'][:60]}...")

    # user selection
    choice = int(input(f"\nEnter product number (0 to {len(results)-1}): "))

    if choice < 0 or choice >= len(results):
          print(f"❌ Invalid choice! Please enter between 0 and {len(results)-1}")
          return

    product_name = results.iloc[choice]['Product']

    print("\n✅ Selected Product:\n", product_name)

    print("\n📝 Reviews:\n")

    reviews = dataset[dataset['Product'] == product_name]['Review'].head(3)

    for r in reviews:
        print("-", r)
        
    recommend(product_name)

In [20]:
product_system("samsung")


🔍 Matching Products:

0 | Samsung | "CLEAR CLEAN ESN" Sprint EPIC 4G Galaxy SPH-D700*FRONT CAMER...
1 | Samsung | ASUS ZenFone 2 (ZE551ML) Unlocked Cellphone,5.5 inch 4GB RAM...
2 | BLU | BLU Zoey 2.4 3G Unlocked Black & Blue Perfect affordable alt...
3 | CNPGD | CNPGD [U.S. Office Extended Warranty] Smartwatch + Unlocked ...
4 | CNPGD | CNPGD [U.S. Office Extended Warranty] Smartwatch + Unlocked ...



Enter product number (0 to 4):  1



✅ Selected Product:
 ASUS ZenFone 2 (ZE551ML) Unlocked Cellphone,5.5 inch 4GB RAM 32GB ROM Android 5.0 Smartphone.(Gray)

📝 Reviews:

- Have had the phone for about two weeks and so far so good! This replaces a 3 year old Nexus 5 that the battery was getting really bad.So far the speed and features of this Zenfone 2 seems as nice if not better, and the price was great. So now its a wait and see if it will hold up and last. Would like to see the OS move up to Marshmallow still 5.

💡 Recommended Products:

Samsung Galaxy J2 J200M 8GB - Factory Unlocked Phone - Retail Packaging - Black
Samsung Galaxy S6 Edge G925A 64GB Unlocked GSM 4G LTE Octa-Core Android Smartphone w/ 16 Megapixel Camera - White
Samsung Galaxy A3 (2016) SM-A310F/DS 16GB Gold, Dual Sim, 4.7", 13MP, Unlocked International Model, No Warranty
Samsung Galaxy J2 SM-J200M/DS Dual Sim LTE 8GB - Black (International Version)
Samsung Galaxy S6 Edge SM-G925F 32GB Factory Unlocked Gold


In [21]:
product_system("motorola"


🔍 Matching Products:

0 | Motorola | AT&T Motorola RAZR V3 No Contract Quad Band GSM Camera Cell ...
1 | Motorola | AT&T Motorola RAZR V3xx No Contract Cell Phone 3G...
2 | Boost Mobile | Boost Mobile Motorola I776W Purple Prepaid...
3 | Consumer Cellular | Consumer Cellular Motorola WX345 Cell Phone...
4 | Motorola | Factory Unlocked Motorola Droid Turbo Special Edition - Ball...



Enter product number (0 to 4):  0



✅ Selected Product:
 AT&T Motorola RAZR V3 No Contract Quad Band GSM Camera Cell Phone Pink

📝 Reviews:

- Not good

💡 Recommended Products:

Motorola RAZR V3m Cell Phone for Verizon with No Contract
Motorola RAZR V3 Unlocked Phone with Camera and Video Player--U.S. Version with Warranty (Pink)
Motorola RAZR V3 Unlocked Phone with Camera and Video Player--International Version with No Warranty (Silver)
Motorola V3I RAZR Cellular Phone ( Unlocked ) - Silver
Motorola RAZR V3 Gold Cellular Phone (Unlocked)


In [24]:
product_system("redmi")


🔍 Matching Products:

0 | Xiaomi | Xiaomi Redmi Note3 Unlocked Cell Phone, 32GbFeatures], LTE F...



Enter product number (0 to 0):  0



✅ Selected Product:
 Xiaomi Redmi Note3 Unlocked Cell Phone, 32GbFeatures], LTE Factory

📝 Reviews:

- Buena

💡 Recommended Products:

Samsung Galaxy Note Edge N915T 32GB Unlocked GSM 4G LTE Cell Phone - White
Samsung Galaxy Note 4 SM-N910F 4G LTE White Factory Unlocked International Model
Samsung Galaxy Note 3 N9006 Factory Unlocked International Version Black
Motorola Droid RAZR 16GB XT912 4G LTE Verizon CDMA Android Phone - White
Samsung Galaxy Note 3 N9000 32GB Unlocked GSM Octa-Core Cell Phone - White


In [22]:
print(product_system("redmiiii"))

❌ No products found
None


In [23]:
import pickle

pickle.dump(dataset, open("products.pkl", "wb"))
pickle.dump(cv, open("vectorizer.pkl", "wb"))
pickle.dump(vectors, open("vectors.pkl", "wb")) 
pickle.dump(vectors, open("vectors.pkl", "wb"))